# Reddit vs Moltbook — Distant Reading & Behavioral Comparison
**Project**: 2026 Spring DH — Digital Humanities 연구
**Generated**: 2026-05-27 · companion to `outline.yaml` / `fields.yaml`

이 노트북은 인간 소셜 네트워크(**Reddit**)와 AI 에이전트 전용 소셜 네트워크(**Moltbook**, 2026-01 출시 · 2026-03 Meta 인수)의 데이터를 **distant reading** + **behavioral analysis** 관점에서 비교하기 위한 작업장이야.

## 노트북 구조
| # | 섹션 | outline.yaml 카테고리 |
|---|---|---|
| 0 | 환경 설정 & 라이브러리 | — |
| 1 | 데이터 로딩 (Reddit + Moltbook) | A, B, C |
| 2 | 공통 스키마 매핑 & 전처리 | C |
| 3 | 텍스트·언어적 차원 | D |
| 4 | 네트워크·구조적 차원 | E |
| 5 | 시간·행동적 차원 | F |
| 6 | 메타데이터·생태계 차원 | G |
| 7 | Agent-Specific Provenance (Moltbook 특화) | I |
| 8 | 종합 비교 (Reddit ↔ Moltbook) | H |
| 9 | 결과 export & 다음 단계 | — |

> 💡 **Tip**: Colab에서 `런타임 → 런타임 유형 변경 → GPU` 켜두면 임베딩/토픽모델링이 빨라져.

---
## 0. 환경 설정 & 라이브러리


In [ ]:
# Colab 환경 감지
import sys, os
IN_COLAB = 'google.colab' in sys.modules
print(f"Colab: {IN_COLAB} | Python: {sys.version.split()[0]}")

In [ ]:
%%capture
# 핵심 라이브러리 설치 (Colab 첫 실행 시 ~3분)
!pip install -q praw zstandard pyarrow datasets
!pip install -q networkx python-louvain scipy scikit-learn
!pip install -q spacy nltk vaderSentiment textstat
!pip install -q sentence-transformers bertopic
!pip install -q seaborn plotly altair
!pip install -q tqdm requests pyyaml
!python -m spacy download en_core_web_sm -q

In [ ]:
# 표준 import
import json, re, gzip, zstandard as zstd, requests, yaml
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

if IN_COLAB:
    PROJECT = Path("/content/dh_reddit_moltbook")
else:
    _here = Path.cwd()
    _root = _here.parent if _here.name == "notebooks" else _here
    PROJECT = _root / "dh_reddit_moltbook"
DATA   = PROJECT / "data"
CACHE  = PROJECT / "cache"
OUT    = PROJECT / "outputs"
for p in (DATA, CACHE, OUT): p.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT)

In [ ]:
# (선택) Google Drive 마운트 — 큰 데이터를 Drive에 저장하고 싶을 때
if IN_COLAB:
    from google.colab import drive
    # drive.mount('/content/drive')   # 필요시 주석 해제
    pass

---
## 1. 데이터 로딩

이 섹션은 outline.yaml의 16개 Items 중 **실제로 노트북에서 바로 시도해볼 수 있는 경로**들만 코드화했어. 큰 dump(Pushshift, Academic Torrents)는 별도 호스트에서 다운로드 후 경로만 잡으면 돼.

> ### ▶ 어떤 셀을 실행할까 (실행 가이드)
> 아래 6개 로더는 **플랫폼당 하나씩만** 고르면 비교가 성립해. 같은 플랫폼 안에서는 서로 대안이야.
>
> | 셀 | 플랫폼 | 역할 | 실행? |
> |---|---|---|---|
> | §1.1 PRAW API | Reddit | 실시간·소량, 인증 필요 | 선택 (빠른 테스트) |
> | §1.2 HF webis-tldr-17 | Reddit | 정제 코퍼스, 인증 불필요. **단 parent_id·timestamp 없음 → 네트워크(§4)·시간(§5) 분석 불가** | 대안 |
> | **§1.3 Academic Torrents 덤프** | Reddit | created_utc·parent_id 완비 → 4개 차원 전부 가능. 2026-Q1 매칭 슬라이스 | **✅ 권장 (메인)** |
> | **§1.4 HF Observatory Archive** | Moltbook | 게시물+메타데이터 메인 소스 | **✅ 권장 (메인)** |
> | §1.5 Public API | Moltbook | 실시간 피드, 아카이브 보충용 | 선택 |
> | §1.6 MoltGraph (.graphml) | Moltbook | 기성 에이전트 상호작용 그래프 (§4를 reply tree 대신 이걸로 할 때만) | 선택 |
>
> **최소 조합: §1.3 + §1.4** → 텍스트·네트워크·시간·커뮤니티 4개 차원 비교가 모두 돌아감.
> 나머지(§1.1 / §1.2 / §1.5 / §1.6)는 빠른 테스트·보강용이므로 건너뛰어도 됨.


### 1.1 Reddit — 공식 API (PRAW · 소량/실시간)

In [ ]:
# [§1.1 · 선택/빠른 테스트] Reddit 공식 API — §1.3을 메인으로 쓰면 건너뛰어도 됨
# Reddit API credentials — https://www.reddit.com/prefs/apps 에서 발급
# script-type app 만든 뒤 client_id / client_secret 채워넣기
REDDIT_CREDS = {
    "client_id":     os.environ.get("REDDIT_CLIENT_ID",     "YOUR_CLIENT_ID"),
    "client_secret": os.environ.get("REDDIT_CLIENT_SECRET", "YOUR_SECRET"),
    "user_agent":    "dh-reddit-moltbook/0.1 by u/yourname",
}

def reddit_client():
    import praw
    return praw.Reddit(**REDDIT_CREDS)

def fetch_subreddit(name, limit=500, kind="new"):
    """단일 서브레딧에서 post 수집 → DataFrame"""
    r = reddit_client()
    sub = r.subreddit(name)
    fn = {"new": sub.new, "hot": sub.hot, "top": sub.top}[kind]
    rows = []
    for p in tqdm(fn(limit=limit), total=limit, desc=name):
        rows.append({
            "id": p.id, "title": p.title, "body": p.selftext,
            "author": str(p.author), "subreddit": name,
            "created_utc": p.created_utc, "score": p.score,
            "n_comments": p.num_comments, "url": p.url,
            "platform": "reddit",
        })
    return pd.DataFrame(rows)

# 예시 — 실제 키를 넣으면 실행 가능
# df_reddit_sample = fetch_subreddit("MachineLearning", limit=200, kind="new")
# df_reddit_sample.head()

### 1.2 Reddit — Hugging Face 정제 코퍼스 (인증 불필요)

In [ ]:
# [§1.2 · 대안] HF 정제 코퍼스 — parent_id/timestamp 없어 네트워크·시간 분석엔 부적합
# webis-tldr-17 — post/summary 4M 쌍, bot-filtered, CC-BY 4.0
from datasets import load_dataset

def load_hf_reddit(name="webis/tldr-17", split="train[:5000]"):
    ds = load_dataset(name, split=split, trust_remote_code=True)
    df = ds.to_pandas()
    df["platform"] = "reddit"
    return df

# df_tldr = load_hf_reddit()
# df_tldr.head()

### 1.3 Reddit — Academic Torrents / Arctic Shift dump (대용량)

`pushshift-style NDJSON.zst` 형식을 읽는 헬퍼. 50GB+ dump는 Drive 또는 Colab Pro+ 디스크에 미리 받아둬.

In [ ]:
# [§1.3 · ✅ 권장 메인] Reddit 덤프 — 4개 차원 전부 가능. 이 셀을 실행
def stream_zst_ndjson(path, max_rows=100_000):
    """zstd-compressed NDJSON dump를 한 줄씩 읽어 list[dict] 반환"""
    dctx = zstd.ZstdDecompressor(max_window_size=2**31)
    rows, buf = [], b""
    with open(path, "rb") as fh, dctx.stream_reader(fh) as r:
        while len(rows) < max_rows:
            chunk = r.read(2**20)
            if not chunk: break
            buf += chunk
            *lines, buf = buf.split(b"\n")
            for ln in lines:
                if not ln.strip(): continue
                try: rows.append(json.loads(ln))
                except: pass
                if len(rows) >= max_rows: break
    return pd.DataFrame(rows)

# --- Reddit dump 소스 (검증됨) ---
# Arctic Shift / Academic Torrents "Reddit comments & submissions" collection.
# Moltbook 대응 기간(2026-01-27~04-14)과 겹치는 2026-Q1 월별 슬라이스만 받는다.
#   torrent infohash : 3d426c47...   (전체 hash는 outline.yaml/pinned_versions에 기록)
#   필요 파일   : RS_2026-01.zst RS_2026-02.zst RS_2026-03.zst RS_2026-04.zst
#                  RC_2026-01.zst RC_2026-02.zst RC_2026-03.zst RC_2026-04.zst
REDDIT_DUMP_INFOHASH = "3d426c47"   # TODO: outline.yaml의 pinned_versions에 full hash 기록
REDDIT_SLICE = ["2026-01", "2026-02", "2026-03", "2026-04"]  # Moltbook 동시기 window

# df_dump = stream_zst_ndjson(DATA / "reddit_dumps" / "RS_2026-01.zst", 50_000)
# df_dump.head()

# --- scripts/download_data.py 로 받은 로컬 ndjson 읽기 (Arctic Shift 산출물) ---
def load_local_reddit(kind="posts"):
    import glob
    base = PROJECT.parent / "data" / "raw" / "reddit"
    frames=[pd.read_json(f, lines=True) for f in glob.glob(str(base / f"*_{kind}.ndjson"))]
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    df["platform"]="reddit"
    return df

df_reddit = load_local_reddit("posts")


### 1.4 Moltbook — GitHub data dump (ExtraE113/moltbook_data)

In [ ]:
# [§1.4 · ✅ 권장 메인] Moltbook Observatory Archive — 이 셀을 실행
# Moltbook 학술 덤프는 GitHub(ExtraE113/moltbook_data) 대신
# Hugging Face Observatory Archive를 기본 소스로 사용 (arXiv:2605.13860).
# revision을 반드시 핀해서 재현 가능성을 확보한다.
MOLTBOOK_HF_REPO = "SimulaMet/moltbook-observatory-archive"
MOLTBOOK_HF_REVISION = None   # TODO: 다운로드 후 commit SHA 기록 -> outline.yaml/pinned_versions

def load_moltbook_archive(split="train", revision=MOLTBOOK_HF_REVISION, max_rows=None):
    """Observatory Archive(HF datasets)를 DataFrame으로 로드"""
    from datasets import load_dataset
    ds = load_dataset(MOLTBOOK_HF_REPO, split=split, revision=revision)
    df = ds.to_pandas() if max_rows is None else ds.select(range(max_rows)).to_pandas()
    df["platform"] = "moltbook"
    print(f"Observatory Archive: {len(df):,} rows  rev={revision or 'HEAD(⚠ 핀 권장)'}")
    return df

# --- (fallback) 원래 GitHub 덤프가 필요할 때만 사용 ---
MOLTBOOK_DUMP_BASE = "https://raw.githubusercontent.com/ExtraE113/moltbook_data/main/"

def fetch_moltbook_dump(filename):
    url = MOLTBOOK_DUMP_BASE + filename
    r = requests.get(url, timeout=60); r.raise_for_status()
    text = r.text.strip()
    data = json.loads(text) if text.startswith("[") else [json.loads(l) for l in text.splitlines() if l.strip()]
    df = pd.DataFrame(data); df["platform"] = "moltbook"
    return df

# df_moltbook = load_moltbook_archive(max_rows=50_000)
# df_moltbook.head()

# --- scripts/download_data.py 로 받은 로컬 parquet 읽기 (오프라인) ---
def load_local_moltbook(config="posts"):
    p = PROJECT.parent / "data" / "raw" / "moltbook" / f"{config}.parquet"
    df = pd.read_parquet(p); df["platform"]="moltbook"
    return df

df_moltbook = load_local_moltbook("posts")


### 1.5 Moltbook — Public API (`POST /posts`)

In [ ]:
# [§1.5 · 선택] 실시간 피드 — 아카이브에 없는 최신 데이터 보충용
# Meta 인수 후 도메인 변경 — 실제 호스트는 www.moltbook.com
MOLTBOOK_API = "https://www.moltbook.com/api/v1"

def fetch_moltbook_feed(limit=200, cursor=None):
    payload = {"limit": limit}
    if cursor: payload["cursor"] = cursor
    headers = {"Authorization": f"Bearer {os.environ.get('MOLTBOOK_TOKEN','')}"}
    r = requests.post(f"{MOLTBOOK_API}/posts", json=payload, headers=headers, timeout=30)
    r.raise_for_status()
    js = r.json()
    df = pd.DataFrame(js.get("data", []))
    df["platform"] = "moltbook"
    return df, js.get("next_cursor")

# df_live, cursor = fetch_moltbook_feed(limit=100)


### 1.6 Moltbook — 학술 데이터셋

`MoltGraph` (arXiv:2603.00646), `Observatory Archive` (arXiv:2605.13860), `MoltNet` (arXiv:2602.13458)은 보통 논문 부록 또는 figshare/Zenodo 링크로 배포돼. 다운로드 후 다음과 같이 로드:

In [ ]:
# [§1.6 · 선택] 기성 에이전트 그래프 — §4를 reply tree 대신 이걸로 할 때만
# MoltGraph (arXiv:2603.00646) — 논문 부록의 figshare/Zenodo 링크에서 .graphml 획득.
# 다운로드 링크와 버전(DOI/revision)을 pinned_versions에 기록해 둔다.
MOLTGRAPH_PATH = DATA / "moltgraph.graphml"
MOLTGRAPH_SOURCE = "arXiv:2603.00646 부록"   # TODO: figshare/Zenodo DOI 확인 후 교체
MOLTGRAPH_VERSION = None                      # TODO: dataset version/DOI 기록

def load_moltgraph(path=MOLTGRAPH_PATH):
    if not Path(path).exists():
        print(f"⚠️ {path} 없음 — {MOLTGRAPH_SOURCE}에서 받아 DATA에 두기")
        return None
    G = nx.read_graphml(path)
    node_attrs = set().union(*(d.keys() for _, d in list(G.nodes(data=True))[:50])) if G.number_of_nodes() else set()
    edge_attrs = set().union(*(d.keys() for *_, d in list(G.edges(data=True))[:50])) if G.number_of_edges() else set()
    print(f"MoltGraph: |V|={G.number_of_nodes():,} |E|={G.number_of_edges():,} "
          f"directed={G.is_directed()}  ver={MOLTGRAPH_VERSION or 'HEAD'}")
    print(f"  node attrs: {sorted(node_attrs)}")
    print(f"  edge attrs: {sorted(edge_attrs)}")
    return G

# G_molt = load_moltgraph()


---
## 2. 공통 스키마 매핑 & 전처리

Reddit과 Moltbook은 필드명이 다르므로 비교 분석을 위해 **canonical schema**로 통일.

In [ ]:
CANONICAL_COLS = [
    "post_id", "author_id", "platform", "community",
    "created_utc", "text", "score", "n_replies",
    "parent_id", "is_bot_or_agent"
]

def to_canonical_reddit(df):
    out = pd.DataFrame()
    out["post_id"]    = df.get("id", df.get("name"))
    out["author_id"]  = df.get("author")
    out["platform"]   = "reddit"
    out["community"]  = df.get("subreddit")
    out["created_utc"]= pd.to_datetime(df.get("created_utc"), unit="s", utc=True)
    title = df.get("title", "").fillna("") if "title" in df else ""
    body  = df.get("body",  df.get("selftext", "")).fillna("")
    out["text"]       = (title + "\n" + body).str.strip()
    out["score"]      = df.get("score", 0)
    out["n_replies"]  = df.get("n_comments", df.get("num_comments", 0))
    out["parent_id"]  = df.get("parent_id")
    out["is_bot_or_agent"] = df.get("author", pd.Series()).astype(str).str.endswith("Bot")
    return out[CANONICAL_COLS]

def to_canonical_moltbook(df):
    out = pd.DataFrame()
    out["post_id"]    = df.get("id")
    out["author_id"]  = df.get("agent_id", df.get("author_id"))
    out["platform"]   = "moltbook"
    out["community"]  = df.get("submolt", df.get("room", df.get("topic")))   # submolt = 커뮤니티
    out["created_utc"]= pd.to_datetime(df.get("created_at", df.get("timestamp")), utc=True)
    title = df["title"].fillna("") if "title" in df.columns else ""          # posts엔 title, comments엔 없음
    body  = df.get("content", df.get("text", "")).fillna("")
    out["text"]       = (title + "\n" + body).str.strip()
    out["score"]      = df.get("score", df.get("likes", df.get("reactions", 0)))
    out["n_replies"]  = df.get("comment_count", df.get("replies", 0))         # posts: comment_count
    out["parent_id"]  = df.get("parent_id")
    out["is_bot_or_agent"] = True  # Moltbook은 기본적으로 모두 agent
    return out[CANONICAL_COLS]

In [ ]:
def basic_clean(text):
    """공통 텍스트 클리닝 — URL, 마크다운, 다중 공백 정리"""
    if not isinstance(text, str): return ""
    text = re.sub(r"https?://\S+", " <URL> ", text)
    text = re.sub(r"\*+|_+|#+", " ", text)        # markdown
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 사용 예시:
df_canon = pd.concat([to_canonical_reddit(df_reddit),
                       to_canonical_moltbook(df_moltbook)], ignore_index=True)
df_canon["text_clean"] = df_canon["text"].map(basic_clean)
df_canon.head()

In [ ]:
# --- MBC-20 transactional 필터 (arXiv:2604.21295) -------------------------------
# Moltbook 글의 상당수는 사람이 읽는 글이 아니라 MBC-20 토큰 inscription /
# launch command / wallet 페이로드다. 비교 분석 전에 분리하고
# discursive 서브셋과 전체 코퍼스 두 가지를 모두 보고한다.
# 주의: 줄머리 앵커(re.M) 대신 페이로드를 전체 검색하고, basic_clean 이전의 raw text에 적용한다
#       (basic_clean이 줄바꿈을 없애 앵커가 깨지는 문제 회피 — 실제 데이터로 검증됨).
MBC20_PATTERNS = re.compile(
    r'"p"\s*:\s*"mbc-?20"'                       # MBC-20 inscription JSON payload
    r'|"op"\s*:\s*"(?:deploy|mint|transfer)"'    # inscription op codes
    r'|!(?:clawnch|lawnchpad|kibu|claw_tech)\b'  # launch commands
    r'|\b0x[a-fA-F0-9]{40}\b'                    # ETH wallet address
    r'|register\s+wallet\s+[a-z0-9]{20,}',       # wallet registration
    re.I,
)

def is_transactional(text):
    return bool(isinstance(text, str) and MBC20_PATTERNS.search(text))

def split_discursive(df, text_col="text"):
    """(discursive_df, transactional_df) 반환. raw 'text' 권장 — JSON 구조 보존돼야 정확."""
    df = df.copy()
    df["is_transactional"] = df[text_col].map(is_transactional)
    return df[~df.is_transactional].copy(), df[df.is_transactional].copy()

# --- 사용 예 + sanity check ---
disc, txn = split_discursive(df_canon[df_canon.platform=="moltbook"])
n = len(disc) + len(txn)
print(f"discursive {len(disc)/n:.1%} | transactional {len(txn)/n:.1%}")
#
# ⚠ 전체 코퍼스 기준 ~63% transactional(arXiv:2604.21295)은 78일 전체 수치다.
#    빠른 샘플은 launch week(2026-01-27~31)에 치우쳐 ~4%만 transactional로 나온다
#    (토큰 inscription 물결은 이후 급증). 전체 비율 재현은 download_data.py --full 사용.


---
## 3. 텍스트·언어적 차원 (D)
- 길이 분포, 어휘 풍부도(TTR), 가독성, 감정/정서 마커
- Anthropomorphism markers (1인칭 비율, 감정 표현 빈도)

In [ ]:
import spacy, textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

# 토크나이저만 필요 → 무거운 컴포넌트 전부 비활성
nlp = spacy.load("en_core_web_sm",
                 disable=["ner","parser","tagger","lemmatizer","attribute_ruler","tok2vec"])

def add_linguistic(df, text_col="text_clean", batch_size=500):
    texts = df[text_col].fillna("").str.slice(0, 5000).tolist()
    rows = []
    for doc, txt in zip(nlp.pipe(texts, batch_size=batch_size), texts):
        toks = [t.text.lower() for t in doc if t.is_alpha]
        n = len(toks) or 1
        fp = sum(1 for t in toks if t in {"i","me","my","mine","myself"})
        rows.append({
            "n_tokens": n, "n_chars": len(txt),
            "ttr": len(set(toks))/n,
            "first_person_rate": fp/n,
            "flesch": textstat.flesch_reading_ease(txt) if txt else 0.0,
            "compound_sent": vader.polarity_scores(txt)["compound"] if txt else 0.0,
        })
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

df_canon = add_linguistic(df_canon)

In [ ]:
# 시각화 — 길이 / 1인칭 비율 / TTR 분포 비교
def plot_linguistic_comparison(df):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = [("n_tokens", "token num (log)", True),
               ("first_person_rate", "first-person-rate", False),
               ("ttr", "Type-Token Ratio", False)]
    for ax, (col, title, logx) in zip(axes, metrics):
        for p, g in df.groupby("platform"):
            data = g[col].dropna()
            if logx: data = np.log1p(data)
            sns.kdeplot(data, ax=ax, label=p, fill=True, alpha=0.4)
        ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()

plot_linguistic_comparison(df_canon)
plt.savefig("../docs/fig_3_linguistic.png", dpi=130, bbox_inches="tight")

In [ ]:
import re

def _mtld_calc(tokens, threshold=0.72):
    factors, types, n = 0.0, set(), 0
    for tok in tokens:
        n += 1; types.add(tok)
        if len(types)/n <= threshold:
            factors += 1; types = set(); n = 0
    if n > 0:
        factors += (1 - len(types)/n) / (1 - threshold)
    return len(tokens)/factors if factors else float(len(tokens))

def mtld(text, threshold=0.72, min_len=50):
    """길이 보정 어휘 다양도. 50토큰 미만은 NaN(신뢰 불가)."""
    toks = re.findall(r"[a-z0-9']+", str(text).lower())
    if len(toks) < min_len:
        return float("nan")
    return (_mtld_calc(toks, threshold) + _mtld_calc(toks[::-1], threshold)) / 2

# 계산 (raw text 기준 — 길이 신호 보존)
df_canon["mtld"] = df_canon["text"].map(mtld)

# 결과 요약
valid = df_canon.dropna(subset=["mtld"])
print("MTLD 계산된 글 수 (≥50토큰):")
print(valid.groupby("platform")["mtld"].agg(["count", "median", "mean"]).round(1))

# 분포 비교
import seaborn as sns, matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
for p in ["moltbook","reddit"]:
    sns.kdeplot(valid.loc[valid.platform==p,"mtld"], label=p, fill=True, alpha=0.4, ax=ax)
ax.set_title("MTLD (길이 보정 어휘 다양도) — 높을수록 다양")
ax.set_xlabel("MTLD"); ax.legend(); plt.tight_layout(); plt.show()

### 3.1 토픽 모델링 (BERTopic) — Reddit vs Moltbook 주제 분포

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

def pool_documents(df, text_col="text_clean", by="thread", min_len=1):
    """Waller & Anderson(2021) 스타일 풀링: 짧은 글을 작성자/스레드 단위로 묶어
    희소성(sparsity)을 줄이고 토픽 안정성을 높인다.
        by='thread'  -> parent_id를 따라 root로 묶음 (parent_id 없으면 post_id)
        by='author'  -> author_id 단위로 묶음
    반환: pooled 텍스트 + 메타(platform, pool_key) DataFrame
    """
    d = df.copy()
    if by == "thread":
        d["_key"] = d["parent_id"].fillna(d["post_id"]).astype(str)
    elif by == "author":
        d["_key"] = d["author_id"].astype(str)
    else:
        raise ValueError("by must be 'thread' or 'author'")
    g = (d.groupby(["platform", "_key"])[text_col]
           .apply(lambda s: " ".join(x for x in s if isinstance(x, str)))
           .reset_index().rename(columns={text_col: "pooled_text", "_key": "pool_key"}))
    g = g[g["pooled_text"].str.split().map(len) >= min_len]
    return g

def fit_bertopic(texts, n_topics="auto", min_topic_size=10):
    embed = SentenceTransformer("all-MiniLM-L6-v2")
    model = BERTopic(embedding_model=embed,
                     min_topic_size=min_topic_size,
                     nr_topics=n_topics, verbose=True)
    topics, _ = model.fit_transform(texts)
    return model, topics

def topic_distribution_by_platform(df, text_col="text_clean", pool_by="thread"):
    # 풀링 단계 추가 — pool_by=None이면 글 단위 그대로
    if pool_by:
        pooled = pool_documents(df, text_col=text_col, by=pool_by)
        texts, base = pooled["pooled_text"].tolist(), pooled
    else:
        texts, base = df[text_col].tolist(), df
    m, topics = fit_bertopic(texts)
    df_t = base.assign(topic=topics)
    pivot = (df_t.groupby(["platform","topic"]).size()
             .unstack(fill_value=0)
             .apply(lambda r: r / r.sum(), axis=1))
    pivot.T.plot(kind="bar", figsize=(14,4),
                 title=f"플랫폼별 토픽 분포 비율 (pool_by={pool_by})")
    plt.tight_layout(); plt.show()
    return m, df_t

# topic_model, df_with_topic = topic_distribution_by_platform(df_canon, pool_by="thread")
# topic_model.get_topic_info().head(15)


---
## 4. 네트워크·구조적 차원 (E)
Reply tree, author/agent mention graph, community 위상 비교.

In [ ]:
import networkx as nx

def build_reply_tree_reddit(posts, comments):
    G = nx.DiGraph()
    for _, r in posts.iterrows():
        G.add_node(r["id"], platform="reddit", author=r.get("author"), kind="post")
    for _, c in comments.iterrows():
        cid = c["id"]; G.add_node(cid, platform="reddit", author=c.get("author"), kind="comment")
        par = c.get("parent_id")
        if isinstance(par, str) and "_" in par:
            par = par.split("_", 1)[1]          # t1_/t3_ 접두사 제거
        if pd.notna(par):
            G.add_edge(par, cid)
    return G

def build_reply_tree_moltbook(posts, comments):
    G = nx.DiGraph()
    for _, r in posts.iterrows():
        G.add_node(r["id"], platform="moltbook", author=r.get("agent_id"), kind="post")
    for _, c in comments.iterrows():
        cid = c["id"]; G.add_node(cid, platform="moltbook", author=c.get("agent_id"), kind="comment")
        par = c.get("parent_id")
        par = par if pd.notna(par) else c.get("post_id")   # 최상위 댓글 → 글에 연결
        if pd.notna(par):
            G.add_edge(par, cid)
    return G

G_reddit = build_reply_tree_reddit(df_reddit, df_reddit_c)
G_molt   = build_reply_tree_moltbook(df_moltbook, df_moltbook_c)
pd.DataFrame({"reddit": reply_tree_stats(G_reddit),
              "moltbook": reply_tree_stats(G_molt)})

In [ ]:
def draw_sample_tree(G, max_nodes=200, title=""):
    comps = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
    nodes = set()
    for c in comps:                       # 큰 스레드부터 채움
        nodes |= c
        if len(nodes) >= max_nodes: break
    H = G.subgraph(list(nodes)[:max_nodes])
    pos = nx.spring_layout(H, seed=42, k=0.5)
    plt.figure(figsize=(8,6))
    nx.draw(H, pos, node_size=20, arrowsize=4, with_labels=False, alpha=0.7)
    plt.title(f"{title} ({H.number_of_nodes()} nodes, {H.number_of_edges()} edges)")
    plt.show()

draw_sample_tree(G_reddit, title="Reddit reply tree (largest threads)")
draw_sample_tree(G_molt,   title="Moltbook reply tree (largest threads)")

In [ ]:
import networkx as nx, pandas as pd, matplotlib.pyplot as plt

def depth_distribution(G):
    lvl = {}
    for d, gen in enumerate(nx.topological_generations(G)):
        for n in gen: lvl[n] = d
    s = pd.Series(lvl); return s[s >= 1]          # depth0=글/고아 제외, 댓글만

def share_by_depth(s, maxd=5):
    return s.clip(upper=maxd).value_counts(normalize=True).sort_index()

tab = pd.DataFrame({"reddit": share_by_depth(depth_distribution(G_reddit)),
                    "moltbook": share_by_depth(depth_distribution(G_molt))}).fillna(0)
tab.index = [f"depth {int(i)}{'+' if i==5 else ''}" for i in tab.index]
print((tab*100).round(1))
tab.plot(kind="bar", figsize=(8,4))
plt.ylabel("share of comments"); plt.title("Reply depth distribution — flat(star) vs nested")
plt.tight_layout(); plt.show()

In [ ]:
import collections
def draw_largest_thread(G, title="", max_nodes=150):
    H = G.subgraph(max(nx.weakly_connected_components(G), key=len))
    roots = [n for n in H if H.in_degree(n) == 0]
    root = max(roots, key=lambda r: len(nx.descendants(H, r))) if roots else list(H)[0]
    depth = {root: 0}; q = collections.deque([root]); seen = {root}
    while q:
        u = q.popleft()
        for v in H.successors(u):
            if v not in seen:
                seen.add(v); depth[v] = depth[u] + 1; q.append(v)
    Hs = H.subgraph(list(seen)[:max_nodes])
    bylvl = collections.defaultdict(list)
    for n in Hs: bylvl[depth[n]].append(n)
    pos = {n: (i - len(ns)/2, -d) for d, ns in bylvl.items() for i, n in enumerate(ns)}
    plt.figure(figsize=(11,5))
    nx.draw(Hs, pos, node_size=30, arrowsize=6, with_labels=False, alpha=0.8, width=0.6)
    plt.title(f"{title} — largest thread (max depth {max(bylvl)}, {Hs.number_of_nodes()} nodes)")
    plt.tight_layout(); plt.show()

draw_largest_thread(G_reddit, "Reddit")
draw_largest_thread(G_molt,   "Moltbook")

In [ ]:
import networkx as nx

meta = df_moltbook.drop_duplicates("id").set_index("id")[
    ["submolt","agent_name","title","content","comment_count"]]

def describe_largest_threads(G, k=5):
    comps = sorted(nx.weakly_connected_components(G), key=len, reverse=True)[:k]
    for c in comps:
        H = G.subgraph(c)
        roots = [n for n in H if H.in_degree(n)==0]
        root = max(roots, key=lambda r: len(nx.descendants(H, r))) if roots else None
        if root in meta.index:
            m = meta.loc[root]
            print(f"[{H.number_of_nodes()} 노드 | 실제 comment_count={m.comment_count}] submolt={m.submolt}")
            print(f"   title: {str(m.title)[:100]}")
            print(f"   {m.agent_name}: {str(m.content)[:180]}".replace(chr(10),' '))
        else:
            print(f"[{H.number_of_nodes()} 노드] 루트가 표본 밖 글 (댓글만 받힘)")
        print()

describe_largest_threads(G_molt)   # 같은 방식으로 G_reddit도

In [ ]:
import numpy as np
fig, ax = plt.subplots(figsize=(9,4))
for p in ["reddit","moltbook"]:
    d = df_canon.loc[df_canon.platform==p, "n_replies"].dropna()
    sns.histplot(np.log1p(d.values), label=f"{p} (n={len(d):,})", bins=40,
                 stat="probability", common_norm=False, alpha=0.45, kde=True, ax=ax)
ax.set_xlabel("log(1 + replies per post)"); ax.set_ylabel("share of posts")
ax.set_title("Replies per post — 집중도 / 알고리즘 부스팅?"); ax.legend()
plt.tight_layout(); plt.show()

def conc(d):
    d = d.sort_values(ascending=False).values; tot = d.sum() or 1
    return pd.Series({"n_posts":len(d), "mean":d.mean(), "median":np.median(d),
                      "max":d.max(), "top1%_share":d[:max(1,len(d)//100)].sum()/tot})
print(pd.DataFrame({p: conc(df_canon.loc[df_canon.platform==p,"n_replies"].dropna())
                    for p in ["reddit","moltbook"]}).round(3))

---
## 5. 시간·행동적 차원 (F)
포스팅 빈도, burst, 일주기 패턴 — 인간(circadian) vs 에이전트(cron/event-driven).

In [ ]:
def add_time_features(df):
    df = df.copy()
    t = pd.to_datetime(df["created_utc"], utc=True)
    df["hour"]    = t.dt.hour
    df["weekday"] = t.dt.day_name()
    df["date"]    = t.dt.date
    return df

def plot_hour_distribution(df):
    df = add_time_features(df)
    pivot = (df.groupby(["platform","hour"]).size()
             .unstack(level=0, fill_value=0)
             .apply(lambda c: c / c.sum()))
    pivot.plot(figsize=(10,4), title="시간대별 포스팅 비율 (UTC) — circadian 흔적 찾기")
    plt.xlabel("hour of day"); plt.ylabel("share of posts")
    plt.tight_layout(); plt.show()

# plot_hour_distribution(df_canon)

In [ ]:
# Burst detection — 단순 z-score 기반
def detect_bursts(df, freq="H", z_thresh=2.5):
    series = (df.assign(t=pd.to_datetime(df.created_utc, utc=True))
                .set_index("t").resample(freq).size())
    z = (series - series.mean()) / series.std()
    bursts = series[z > z_thresh]
    return series, bursts

def plot_bursts(df, freq="H"):
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    for ax, (p, g) in zip(axes, df.groupby("platform")):
        s, b = detect_bursts(g, freq=freq)
        s.plot(ax=ax, label="rate")
        ax.scatter(b.index, b.values, color="red", label="burst", zorder=5)
        ax.set_title(f"{p} — posting rate ({freq})"); ax.legend()
    plt.tight_layout(); plt.show()

# plot_bursts(df_canon, freq="H")

In [ ]:
import numpy as np, matplotlib.pyplot as plt

dft = add_time_features(df_canon)   # hour/weekday/date 추가

# 1) 시간대별 포스팅 비율
pivot = (dft.groupby(["platform","hour"]).size()
            .unstack(level=0, fill_value=0)
            .apply(lambda c: c / c.sum()))   # 플랫폼별 합=1로 정규화
ax = pivot.plot(figsize=(11,4), marker="o",
                title="시간대별 포스팅 비율 (UTC) — 인간의 수면 리듬 vs AI의 24시간 균일성")
ax.set_xlabel("hour of day (UTC)"); ax.set_ylabel("share of posts")
ax.set_xticks(range(0,24)); plt.tight_layout(); plt.show()

# 2) circadian "평평함" 정량화 (낮을수록 봇처럼 균일)
print("플랫폼별 시간대 분포 통계 (균일=1/24≈0.0417):")
for p in pivot.columns:
    s = pivot[p]
    cv = s.std() / s.mean()                       # 변동계수
    ratio = s.max() / s.min() if s.min()>0 else np.inf  # 피크/저점 비
    peak, trough = s.idxmax(), s.idxmin()
    print(f"  {p:9} CV={cv:.2f}  peak/trough={ratio:.1f}배  "
          f"(피크 {peak}시 {s.max():.1%} / 저점 {trough}시 {s.min():.1%})")

In [ ]:
m = dft[dft.platform=="moltbook"]
print("날짜별 글 수:")
print(m.groupby("date").size(), "\n")
print("(날짜,시간) 상위 10 — 스파이크가 어디서 오는지:")
print(m.groupby(["date","hour"]).size().sort_values(ascending=False).head(10))

---
## 6. 메타데이터·생태계 차원 (G)
커뮤니티 규모, 작성자/에이전트 활동 분포, 봇 attribution.

In [ ]:
def author_activity_distribution(df):
    counts = df.groupby(["platform","author_id"]).size()
    platforms = list(counts.index.get_level_values(0).unique())
    fig, axes = plt.subplots(1, len(platforms), figsize=(12,4))
    if len(platforms) == 1: axes = [axes]
    palette = {"moltbook": "C0", "reddit": "C1"}
    for ax, p in zip(axes, platforms):
        vals = np.log1p(counts.loc[p].values)
        sns.histplot(vals, ax=ax, bins=40, kde=True,
                     color=palette.get(p, "C2"), alpha=0.6)
        ax.set_title(f"{p}  (n_authors={counts.loc[p].size:,})")
        ax.set_xlabel("log(1 + posts per author)")
        ax.set_ylabel("count")
    fig.suptitle("Author/agent activity distribution — power-law vs uniform?")
    plt.tight_layout(); plt.show()

author_activity_distribution(df_canon)

In [ ]:
def author_activity_distribution(df):
    counts = df.groupby(["platform","author_id"]).size()
    fig, ax = plt.subplots(figsize=(9,4))
    for p in counts.index.get_level_values(0).unique():
        vals = np.log1p(counts.loc[p].values)
        sns.histplot(vals, label=f"{p} (n={counts.loc[p].size:,})",
                     ax=ax, alpha=0.45, kde=True, bins=40,
                     stat="probability", common_norm=False)   # 각 플랫폼 합=1
    ax.set_xlabel("log(1 + posts per author)")
    ax.set_ylabel("share of authors")
    ax.set_title("Author/agent activity distribution — power-law vs uniform?")
    ax.legend(); plt.tight_layout(); plt.show()

author_activity_distribution(df_canon)

In [ ]:
# 댓글 로드 (4월 윈도우, posts와 동일 기간)
df_reddit_c   = load_local_reddit("comments")
df_moltbook_c = load_local_moltbook("comments")

# posts + comments 를 canonical로 합쳐 "총 기여" 코퍼스 구성
contrib = pd.concat([
    to_canonical_reddit(df_reddit),      # reddit posts
    to_canonical_reddit(df_reddit_c),    # reddit comments
    to_canonical_moltbook(df_moltbook),  # moltbook posts
    to_canonical_moltbook(df_moltbook_c) # moltbook comments
], ignore_index=True)

# 1) 작성자당 평균 기여 수 (글+댓글)
summary = (contrib.groupby("platform")
           .agg(contributions=("post_id", "count"),
                authors=("author_id", "nunique"))
           .assign(per_author=lambda d: d.contributions / d.authors))
print("총 기여(post+comment) 기준:")
print(summary.round(2))

# 2) 분포 비교 (플랫폼별 퍼센트 정규화)
counts = contrib.groupby(["platform", "author_id"]).size()
fig, ax = plt.subplots(figsize=(9,4))
for p in counts.index.get_level_values(0).unique():
    vals = np.log1p(counts.loc[p].values)
    sns.histplot(vals, label=f"{p} (n={counts.loc[p].size:,})",
                 ax=ax, alpha=0.45, kde=True, bins=40,
                 stat="probability", common_norm=False)
ax.set_xlabel("log(1 + total contributions per author)")
ax.set_ylabel("share of authors")
ax.set_title("Author/agent total activity (posts + comments)")
ax.legend(); plt.tight_layout(); plt.show()

---
## 7. Agent-Specific Provenance (I) — Moltbook 특화

`model_provenance`, `prompt_template_traces`, `anthropomorphism_markers`, `human_in_the_loop_ratio` 등 Moltbook contamination(reverse-CAPTCHA 우회) 추정.

In [ ]:
# Moltbook agent의 모델 provenance 추정 — (a) 명시적 column, (b) 자기소개 문자열 정규식
# self-intro 예: "I am Claude, an AI made by Anthropic", "powered by gpt-4o", "model: claude-3.5-sonnet"
MODEL_PATTERNS = [
    ("claude",   re.compile(r"\bclaude(?:[-\s]?(?:ai|opus|sonnet|haiku|[0-9.]+))?\b|\banthropic\b", re.I)),
    ("gpt",      re.compile(r"\bgpt[-\s]?[0-9o.]+\b|\bopenai\b|\bchatgpt\b", re.I)),
    ("gemini",   re.compile(r"\bgemini\b|\bgoogle\s+deepmind\b|\bpalm\b", re.I)),
    ("llama",    re.compile(r"\bllama[-\s]?[0-9.]*\b|\bmeta\s+ai\b", re.I)),
    ("mistral",  re.compile(r"\bmistral\b|\bmixtral\b", re.I)),
    ("qwen",     re.compile(r"\bqwen[-\s]?[0-9.]*\b", re.I)),
    ("grok",     re.compile(r"\bgrok[-\s]?[0-9.]*\b", re.I)),
]

def infer_model_from_text(text):
    """자기소개/서명 문자열에서 모델 family 추출 (첫 매치)"""
    if not isinstance(text, str): return None
    for fam, pat in MODEL_PATTERNS:
        if pat.search(text): return fam
    return None

def model_provenance_distribution(df_molt, text_cols=("bio", "description", "text", "content")):
    # (a) 명시적 column 우선
    col = next((c for c in ["model","llm","model_family","backend"] if c in df_molt.columns), None)
    if col is not None:
        series = df_molt[col].astype(str)
        source = f"column:{col}"
    else:
        # (b) 텍스트 정규식 fallback
        tcol = next((c for c in text_cols if c in df_molt.columns), None)
        if tcol is None:
            print("⚠️ model provenance column/text 모두 없음 — dump 스키마 확인 필요")
            return None
        series = df_molt[tcol].map(infer_model_from_text)
        source = f"regex:{tcol}"
    vc = series.dropna().value_counts().head(15)
    print(f"provenance source = {source};  identified {vc.sum():,}/{len(df_molt):,}")
    vc.plot(kind="barh", figsize=(8,4), title=f"Moltbook agent model provenance ({source})")
    plt.tight_layout(); plt.show()
    return vc

model_provenance_distribution(df_moltbook)


In [ ]:
df_agents = load_local_moltbook("agents")
print("agents:", df_agents.shape)
model_provenance_distribution(df_agents)   # description(bio) 자동 우선 → regex:description

In [ ]:
RUNTIME = re.compile(r"\b(clawdbot|openclaw|claw[_\s-]?tech|claw\b)", re.I)
df_agents["runtime_claude_like"] = df_agents["description"].fillna("").str.contains(RUNTIME)
print("Clawdbot/Claw 계열 런타임 언급:",
      f"{df_agents['runtime_claude_like'].mean():.1%} ({df_agents['runtime_claude_like'].sum():,}명)")

In [ ]:
# 의심스러운 human contamination 탐지 — 매우 거친 휴리스틱
HUMAN_HINTS = re.compile(r"\b(lol|lmao|brb|tbh|imo|gn|gm|fr fr|ngl)\b", re.I)
LATE_NIGHT_HOURS = set(range(2, 6))

def human_contamination_score(row):
    s = 0
    if HUMAN_HINTS.search(row.text or ""): s += 1
    if pd.to_datetime(row.created_utc, utc=True).hour in LATE_NIGHT_HOURS: s += 1
    if row.text and len(row.text) > 800: s += 1
    return s

def add_contamination(df_molt):
    df_molt = df_molt.copy()
    df_molt["contam_score"] = df_molt.apply(human_contamination_score, axis=1)
    return df_molt

df_molt2 = add_contamination(df_canon[df_canon.platform=="moltbook"])
print("Suspected human-contaminated (score>=2):",
       (df_molt2.contam_score>=2).mean())

---
## 8. 종합 비교 (H) — Reddit ↔ Moltbook 한눈에 보기

In [ ]:
def summary_table(df):
    rows = []
    for p, g in df.groupby("platform"):
        rows.append({
            "platform": p,
            "n_posts": len(g),
            "n_authors": g["author_id"].nunique(),
            "n_communities": g["community"].nunique(),
            "median_len": g["text"].str.len().median(),
            "median_replies": g["n_replies"].median(),
            "median_score": g["score"].median(),
        })
    return pd.DataFrame(rows).set_index("platform")

summary_table(df_canon)

In [ ]:
# 통계 검정 — 두 플랫폼 분포가 다른지 확인
from scipy import stats

def compare_distributions(df, col):
    a = df.loc[df.platform=="reddit",  col].dropna()
    b = df.loc[df.platform=="moltbook", col].dropna()
    if len(a)<2 or len(b)<2: return None
    ks = stats.ks_2samp(a, b)
    mw = stats.mannwhitneyu(a, b, alternative="two-sided")
    return {"KS_stat": ks.statistic, "KS_p": ks.pvalue,
            "MW_U": mw.statistic, "MW_p": mw.pvalue,
            "mean_diff": a.mean() - b.mean()}

pd.DataFrame({c: compare_distributions(df_canon, c)
               for c in ["n_tokens","ttr","first_person_rate","compound_sent"]
               if c in df_canon}).T

In [ ]:
# 종합 dashboard — radar chart
import matplotlib.pyplot as plt

def radar_compare(df_summary):
    metrics = df_summary.select_dtypes(include=[np.number]).columns
    vals = df_summary[metrics]
    vals_norm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
    for p, row in vals_norm.iterrows():
        v = row.tolist() + [row.tolist()[0]]
        ax.plot(angles, v, label=p); ax.fill(angles, v, alpha=0.2)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics, fontsize=9)
    ax.set_yticklabels([]); ax.legend(loc="upper right")
    ax.set_title("Reddit ↔ Moltbook 정규화 비교")
    plt.tight_layout(); plt.show()

radar_compare(summary_table(df_canon))

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats

# ───── 1) 플랫폼 요약표 ─────
def summary_table(df):
    rows = []
    for p, g in df.groupby("platform"):
        rows.append({
            "platform": p,
            "n_posts": len(g),
            "n_authors": g.author_id.nunique(),
            "posts_per_author": round(len(g) / g.author_id.nunique(), 2),
            "median_tokens": round(g["n_tokens"].median(), 1) if "n_tokens" in g else np.nan,
            "median_mtld": round(g["mtld"].median(), 1) if "mtld" in g else np.nan,
            "mean_first_person": round(g["first_person_rate"].mean(), 4) if "first_person_rate" in g else np.nan,
            "mean_sentiment": round(g["compound_sent"].mean(), 3) if "compound_sent" in g else np.nan,
            "mean_replies": round(g["n_replies"].mean(), 2),
            "median_score": g["score"].median(),
        })
    return pd.DataFrame(rows).set_index("platform")

summ = summary_table(df_canon)
print("=== 요약표 ==="); print(summ.T)

# ───── 2) 분포 차이 통계검정 (Reddit vs Moltbook) ─────
def compare_distributions(df, cols):
    out = {}
    for col in cols:
        if col not in df.columns: continue
        a = df.loc[df.platform == "reddit",   col].dropna()
        b = df.loc[df.platform == "moltbook", col].dropna()
        if len(a) < 2 or len(b) < 2: continue
        ks = stats.ks_2samp(a, b)
        mw = stats.mannwhitneyu(a, b, alternative="two-sided")

---
## 9. 결과 export & 다음 단계

In [ ]:
# 분석 결과를 outputs/ 에 저장
def export_all(df_canon, summary_df, out_dir=OUT):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    df_canon.to_parquet(out_dir / "canonical_posts.parquet", index=False)
    summary_df.to_csv(out_dir / "summary_table.csv")
    print(f"saved → {out_dir}")

# export_all(df_canon, summary_table(df_canon))

In [ ]:
# %load ../scripts/export_results.py
"""
슬라이드용 데이터 CSV 내보내기.

사용법: 노트북에서 §1~§8 분석 셀을 모두 실행해 아래 변수들이 메모리에 있는 상태에서,
        이 파일 내용을 노트북 마지막 셀에 붙여 실행한다(또는 %load scripts/export_results.py).

필요한 in-memory 변수:
  df_canon (피처 포함: n_tokens, ttr, first_person_rate, compound_sent, mtld, n_replies, score,
            created_utc, platform, author_id, community, parent_id, post_id)
  df_reddit, df_reddit_c, df_moltbook, df_moltbook_c, df_agents
  G_reddit, G_molt                       (§4 reply 그래프)
  to_canonical_reddit, to_canonical_moltbook, infer_model_from_text  (노트북 정의 함수)

출력: <repo>/presentation/data/*.csv  (UTF-8-SIG, 엑셀에서도 한글 OK)
"""
import numpy as np, pandas as pd

OUT = PROJECT.parent / "presentation" / "data"          # PROJECT = 노트북 §0에서 정의됨
OUT.mkdir(parents=True, exist_ok=True)

def save(df, name):
    p = OUT / name
    df.to_csv(p, index=False, encoding="utf-8-sig")
    print("✓", name, tuple(df.shape))

# 1) 요약표 ------------------------------------------------------------------
rows = []
for p, g in df_canon.groupby("platform"):
    rows.append(dict(platform=p, n_posts=len(g), n_authors=g.author_id.nunique(),
        posts_per_author=round(len(g)/g.author_id.nunique(), 3),
        median_tokens=g["n_tokens"].median() if "n_tokens" in g else np.nan,
        median_mtld=g["mtld"].median() if "mtld" in g else np.nan,
        mean_first_person=g["first_person_rate"].mean() if "first_person_rate" in g else np.nan,
        mean_sentiment=g["compound_sent"].mean() if "compound_sent" in g else np.nan,
        mean_replies=g["n_replies"].mean(), median_score=g["score"].median()))
save(pd.DataFrame(rows), "summary.csv")

# 2) 피처 분포 히스토그램(tidy: feature, platform, bin_center, share) --------
def hist_tidy(df, col, feature, bins=30, log=False):
    out = []
    # 공통 빈 경계: 두 플랫폼 전체 값으로 edges를 한 번만 계산한다.
    # (이전 버전은 플랫폼마다 따로 np.histogram을 호출해 빈 폭이 달라졌고,
    #  y축이 share라서 빈 폭이 넓은 쪽의 면적이 부당하게 커 보이는 문제가 있었음)
    v_all = df[col].dropna().values
    if log: v_all = np.log1p(v_all)
    if len(v_all) == 0: return pd.DataFrame(out)
    edges = np.histogram_bin_edges(v_all, bins=bins)
    ctr = (edges[:-1] + edges[1:]) / 2
    for p, g in df.groupby("platform"):
        v = g[col].dropna().values
        if log: v = np.log1p(v)
        if len(v) == 0: continue
        cnt, _ = np.histogram(v, bins=edges)   # 공통 edges 사용 → 빈 폭 통일
        sh = cnt / cnt.sum()
        for c, s in zip(ctr, sh):
            out.append(dict(feature=feature, platform=p, bin_center=round(float(c), 4), share=round(float(s), 5)))
    return pd.DataFrame(out)

feat = []
if "n_tokens" in df_canon:          feat.append(hist_tidy(df_canon, "n_tokens", "log_n_tokens", log=True))
if "ttr" in df_canon:               feat.append(hist_tidy(df_canon, "ttr", "ttr"))
if "first_person_rate" in df_canon: feat.append(hist_tidy(df_canon, "first_person_rate", "first_person_rate"))
if "compound_sent" in df_canon:     feat.append(hist_tidy(df_canon, "compound_sent", "sentiment"))
if "mtld" in df_canon:              feat.append(hist_tidy(df_canon.dropna(subset=["mtld"]), "mtld", "mtld"))
if feat: save(pd.concat(feat, ignore_index=True), "feature_hist.csv")

# 3) circadian (시간대별 share) + 통계 --------------------------------------
tmp = df_canon.copy()
tmp["hour"] = pd.to_datetime(tmp["created_utc"], utc=True).dt.hour
piv = tmp.groupby(["platform", "hour"]).size().rename("n").reset_index()
piv["share"] = piv.groupby("platform")["n"].transform(lambda s: s / s.sum())
save(piv[["platform", "hour", "share"]], "circadian.csv")
crows = []
for p, g in piv.groupby("platform"):
    s = g.set_index("hour")["share"]
    crows.append(dict(platform=p, cv=round(s.std()/s.mean(), 3),
        peak_hour=int(s.idxmax()), peak_share=round(float(s.max()), 4),
        trough_hour=int(s.idxmin()), trough_share=round(float(s.min()), 4)))
save(pd.DataFrame(crows), "circadian_stats.csv")

# 4) 작성자 활동(글+댓글) ----------------------------------------------------
try:
    contrib = pd.concat([to_canonical_reddit(df_reddit), to_canonical_reddit(df_reddit_c),
                         to_canonical_moltbook(df_moltbook), to_canonical_moltbook(df_moltbook_c)],
                        ignore_index=True)
    cnt = contrib.groupby(["platform", "author_id"]).size().rename("c").reset_index()
    asum = cnt.groupby("platform").agg(n_authors=("author_id", "nunique"), total=("c", "sum")).reset_index()
    asum["per_author"] = round(asum["total"] / asum["n_authors"], 3)
    save(asum, "author_summary.csv")
    ha = []
    for p, g in cnt.groupby("platform"):
        v = np.log1p(g["c"].values); c2, e = np.histogram(v, bins=30); sh = c2 / c2.sum()
        ctr = (e[:-1] + e[1:]) / 2
        for cc, ss in zip(ctr, sh):
            ha.append(dict(platform=p, log_bin_center=round(float(cc), 4), share=round(float(ss), 5)))
    save(pd.DataFrame(ha), "author_activity_hist.csv")
except NameError as e:
    print("· author activity 스킵 (댓글 df 필요):", e)

# 5) reply 깊이 분포 ---------------------------------------------------------
try:
    import networkx as nx
    def depth_share(G):
        lvl = {}
        for d, gen in enumerate(nx.topological_generations(G)):
            for n in gen: lvl[n] = d
        s = pd.Series(lvl); s = s[s >= 1].clip(upper=5)
        return s.value_counts(normalize=True).sort_index()
    dd = []
    for name, G in [("reddit", G_reddit), ("moltbook", G_molt)]:
        for d, sh in depth_share(G).items():
            dd.append(dict(platform=name, depth=f"{int(d)}{'+' if d == 5 else ''}", share=round(float(sh), 5)))
    save(pd.DataFrame(dd), "reply_depth.csv")
except NameError as e:
    print("· reply depth 스킵 (G_reddit/G_molt 필요):", e)

# 6) 글당 답글 수 / 주목 집중도 ----------------------------------------------
rr = []
for p, g in df_canon.groupby("platform"):
    d = g["n_replies"].dropna().sort_values(ascending=False).values
    tot = d.sum() or 1
    rr.append(dict(platform=p, n_posts=len(d), mean=round(float(d.mean()), 3),
        median=float(np.median(d)), max=int(d.max()),
        top1pct_share=round(float(d[:max(1, len(d)//100)].sum()/tot), 4)))
save(pd.DataFrame(rr), "replies_per_post.csv")

# 7) 모델 provenance ---------------------------------------------------------
try:
    pv = []
    for m, c in df_moltbook["content"].fillna("").map(infer_model_from_text).dropna().value_counts().items():
        pv.append(dict(source="post_content", model=m, count=int(c)))
    for m, c in df_agents["description"].fillna("").map(infer_model_from_text).dropna().value_counts().items():
        pv.append(dict(source="agent_bio", model=m, count=int(c)))
    save(pd.DataFrame(pv), "provenance.csv")
except NameError as e:
    print("· provenance 스킵 (df_agents/infer_model_from_text 필요):", e)

# 8) 분포 차이 검정 ----------------------------------------------------------
try:
    from scipy import stats
    st = []
    for col in ["n_tokens", "ttr", "first_person_rate", "compound_sent", "mtld", "n_replies"]:
        if col not in df_canon: continue
        a = df_canon.loc[df_canon.platform == "reddit", col].dropna()
        b = df_canon.loc[df_canon.platform == "moltbook", col].dropna()
        if len(a) < 2 or len(b) < 2: continue
        ks = stats.ks_2samp(a, b); mw = stats.mannwhitneyu(a, b, alternative="two-sided")
        st.append(dict(feature=col, ks_stat=round(ks.statistic, 4), ks_p=ks.pvalue, mw_p=mw.pvalue,
            mean_reddit=round(float(a.mean()), 4), mean_moltbook=round(float(b.mean()), 4)))
    save(pd.DataFrame(st), "stat_tests.csv")
except Exception as e:
    print("· stat tests 스킵:", e)

print("\n완료 →", OUT)


### 🔭 다음 단계 체크리스트

- [ ] `/research-deep` 결과 JSON을 받아서 → 각 데이터 소스의 실제 다운로드 URL과 스키마를 1.x 셀에 채워넣기
- [ ] Reddit 측: Arctic Shift에서 최근 6개월 일부 서브레딧만 뽑아 작은 비교군 만들기
- [ ] Moltbook 측: `ExtraE113/moltbook_data` README의 실제 파일명을 1.4 셀에 반영
- [ ] Section 3 토픽모델링 결과를 starting point의 "4차원 분석 프레임"과 매핑
- [ ] Section 5 burst 결과로 인간 circadian vs 에이전트 cron 패턴 가설 검정
- [ ] Section 7 contamination score 분포로 Moltbook 데이터의 신뢰 구간 결정
- [ ] 최종 비교 결과를 `report.md`로 정리 (또는 `/research-report` 활용)

> 본 노트북의 모든 셀은 **데이터 없이도 정의만 로드**되며, 실제 API 키/dump 경로만 채우면 바로 실행 가능.